# Judge distillation — QLoRA on Qwen3-8B (Phase 4)

Trains the critique-judge task ("is this candidate finding genuine?") into Qwen3-8B,
so the pass can drop from the 35B to the 8B in production.

**Runtime:** Colab GPU (T4 16GB is enough). `Runtime → Change runtime type → T4 GPU`.

**Inputs:** upload `train.jsonl` and `eval.jsonl` from
`backend/evals/distill/data/dataset/` when prompted below.

**Outputs:** LoRA adapter zip (and optionally a merged GGUF for Ollama).

Data format: each row is `{"messages": [user, assistant]}` where the user turn is the
PRODUCTION judge prompt and the assistant turn is rationale-first JSON. Training masks
the user turn (loss on the response only). Chat template runs with thinking disabled,
matching production (`think=False`).

In [ ]:
%pip install -q unsloth
import torch
print(torch.cuda.get_device_name(0))

In [ ]:
# Upload train.jsonl and eval.jsonl
from google.colab import files
up = files.upload()
assert 'train.jsonl' in up and 'eval.jsonl' in up, 'upload both files'

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 4096  # longest prompt ~2.7k tokens + completion
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3-8B-unsloth-bnb-4bit',  # fallback: 'unsloth/Qwen3-8B'
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=41,
)

In [ ]:
import json
from datasets import Dataset

def load_rows(path):
    return [json.loads(l) for l in open(path, encoding='utf-8')]

def to_text(row):
    # thinking disabled to match production (Ollama think=False)
    return tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False,
        enable_thinking=False,
    )

train_rows = load_rows('train.jsonl')
train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_rows])
print(len(train_ds), 'train examples')
print(train_ds[0]['text'][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type='linear',
        logging_steps=10,
        optim='adamw_8bit',
        seed=41,
        output_dir='out',
        report_to='none',
    ),
)
# loss on assistant turns only
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
trainer.train()

In [ ]:
# Evaluation: per-category precision/recall on eval.jsonl.
# Set EVAL_BASE=True on a fresh runtime (before training) to get the
# untuned baseline for comparison.
import re, collections
EVAL_BASE = False

FastLanguageModel.for_inference(model)
eval_rows = load_rows('eval.jsonl')

def judge(user_content):
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    ids = tokenizer(text, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=160, temperature=0.1, do_sample=False)
    reply = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'"genuine"\s*:\s*(true|false)', reply)
    return (m.group(1) == 'true') if m else None, reply

def category_of(user_content):
    m = re.search(r'Proposed finding \(([^)]+)\)', user_content)
    return m.group(1) if m else '?'

stats = collections.defaultdict(lambda: {'tp':0,'fp':0,'fn':0,'tn':0,'unparsed':0})
for i, row in enumerate(eval_rows):
    user = row['messages'][0]['content']
    gold = json.loads(re.sub(r'^```json\n|\n```$', '', row['messages'][1]['content']))['genuine']
    pred, _ = judge(user)
    cat = category_of(user)
    for key in (cat, 'TOTAL'):
        s = stats[key]
        if pred is None: s['unparsed'] += 1
        elif pred and gold: s['tp'] += 1
        elif pred and not gold: s['fp'] += 1
        elif not pred and gold: s['fn'] += 1
        else: s['tn'] += 1
    if (i+1) % 25 == 0: print(f'{i+1}/{len(eval_rows)}')

print(f"\n{'category':28s} {'P':>6s} {'R':>6s} {'n':>5s} {'unparsed':>9s}")
for cat, s in sorted(stats.items()):
    n = s['tp']+s['fp']+s['fn']+s['tn']+s['unparsed']
    p = s['tp']/max(s['tp']+s['fp'],1); r = s['tp']/max(s['tp']+s['fn'],1)
    print(f'{cat:28s} {p:6.2f} {r:6.2f} {n:5d} {s["unparsed"]:9d}')

In [ ]:
# Save + download the adapter
model.save_pretrained('judge-adapter')
tokenizer.save_pretrained('judge-adapter')
!zip -qr judge-adapter.zip judge-adapter
files.download('judge-adapter.zip')

## Optional: merged GGUF for Ollama

Run the next cell only if eval numbers justify deployment. It merges the adapter
and exports a q4_k_m GGUF (~5GB download), then serve locally with:

```
# Modelfile
FROM ./judge-q4_k_m.gguf
```
`ollama create qwen3-8b-judge -f Modelfile` — then point the audit role's
fast-model env at `qwen3-8b-judge` for the critique judge and re-run
`micro_runner --pass critique` on the dev set before trusting it.

In [ ]:
# model.save_pretrained_gguf('judge-gguf', tokenizer, quantization_method='q4_k_m')
# files.download('judge-gguf/unsloth.Q4_K_M.gguf')